## Libraries and Imports

In [2]:
from datetime import date, timedelta
import polars as pl

from dataframe.shore_handling import salt
from dataframe.transport import forklift

## Logistics forklift record

In [3]:
# Dropbox Path

forklift_path: str = (
    r"C:\Users\gmounac\Dropbox\Container and Transport\Transport Section\Forklift Usage\Forklift Record.xlsx"
)

forklift_sheet: str = "Forklift_Operation"

In [4]:
# DataFrame filtered to current year
CURRENT_YEAR = date.today().year

forklift_df = (
    pl.read_excel(
        forklift_path,
        sheet_name=forklift_sheet,
        engine="calamine",
        schema_overrides={"Time Out": pl.Time, "Time In": pl.Time, "Duration": pl.Time},
    ).filter(pl.col("Date of Service").dt.year().eq(CURRENT_YEAR))
    .lazy()
)

#

In [5]:
forklift_df.with_columns(
    pl.col("Vessel/Client")
    .str.to_uppercase()).filter(pl.col("Date of Service").eq(date(2025,7,16))).collect()

Date of Service,Driver,Forklift No.,Time Out,Time In,Duration,Vessel/Client,Purpose,Invoiced in:,Column1
date,str,str,time,time,time,str,str,str,null
2025-07-16,"""Bryan""","""F5""",08:03:00,08:09:00,00:06:00,"""ALAKRANA""","""Ship Supplies""","""ECHEBASTAR""",null
2025-07-16,"""Bryan""","""F5""",08:23:00,08:28:00,00:05:00,"""ALAKRANA""","""Ship Supplies""","""ECHEBASTAR""",null
2025-07-16,"""Bryan""","""F5""",09:49:00,09:51:00,00:02:00,"""ALAKRANA""","""ship Supplies""","""ECHEBASTAR""",null
2025-07-16,"""Versange""","""F7""",10:58:00,11:06:00,00:08:00,"""ALAKRANA""","""Ship Supplies""","""ECHEBASTAR""",null
2025-07-16,"""Versange""","""F7""",11:36:00,11:43:00,00:07:00,"""ALAKRANA""","""Ship Supplies""","""ECHEBASTAR""",null
…,…,…,…,…,…,…,…,…,…
2025-07-16,"""Bryan""","""F5""",16:11:00,16:19:00,00:08:00,"""NOUR""","""Ship Supplies""","""STO""",null
2025-07-16,"""Bryan""","""F5""",16:43:00,17:36:00,00:53:00,"""TXORI BERRI""","""Salt loading""","""STO""",null
2025-07-16,"""Bryan""","""F5""",17:38:00,18:47:00,01:09:00,"""ALAKRANA""","""Shifting boat supplies""","""ECHEBASTAR""",null


In [6]:
# Latest records

print("Lastest Record")

forklift_df.select(pl.col("Date of Service").max()).collect()

Lastest Record


Date of Service
date
2025-11-24


In [7]:
print("Monthly hours")


forklift_df.group_by(pl.col("Date of Service").dt.month().alias("month")).agg(
    pl.col("Duration").cast(pl.Duration).dt.total_minutes().sum() / 60
).sort(by="month").with_columns(
    dv=(pl.col("Duration") - 65).round().cast(pl.Int32)
).collect()

Monthly hours


month,Duration,dv
i8,f64,i32
1,68.616667,4
2,81.716667,17
3,79.3,14
4,72.833333,8
5,71.516667,7
…,…,…
7,96.416667,31
8,79.35,14
9,95.333333,30


## Compare Forklift Records with the Salt Records

In [8]:
forklift_salt_log = forklift_df.filter(
    pl.col("Purpose")
    .eq(pl.lit("Salt loading"))
    .or_(pl.col("Purpose").eq(pl.lit("loading salt")))
).with_columns(
    pl.col("Vessel/Client").str.to_uppercase().str.replace("GALERNA II", "GALERNA DOS")
)

In [13]:
forklift_salt_log.collect()

Date of Service,Driver,Forklift No.,Time Out,Time In,Duration,Vessel/Client,Purpose,Invoiced in:,Column1
date,str,str,time,time,time,str,str,str,null
2025-01-03,"""Christopher""","""F5""",10:20:00,11:52:00,01:32:00,"""ATERPE ALAI""","""Salt loading""","""ECHEBASTAR""",null
2025-01-06,"""Christopher""","""F5""",17:20:00,18:50:00,01:30:00,"""DOLOMIEU""","""Salt loading""","""SAPMER - UPSTREAM""",null
2025-01-14,"""Christopher""","""F6""",17:20:00,17:45:00,00:25:00,"""FRANCHE TERRE""","""Salt loading""","""SAPMER - UPSTREAM""",null
2025-01-20,"""Christopher""","""F4""",17:50:00,23:00:00,05:10:00,"""IZURDIA""","""Salt loading""","""STO""",null
2025-01-21,"""Christopher""","""F4""",17:45:00,22:30:00,04:45:00,"""IZURDIA""","""Salt loading""","""STO""",null
…,…,…,…,…,…,…,…,…,…
2025-10-18,"""Versange""","""F7""",18:00:00,23:00:00,05:00:00,"""PACIFIC STAR""","""Salt loading""","""STO""",null
2025-11-17,"""Mike""","""F6""",17:30:00,20:30:00,03:00:00,"""ALAKRANA""","""Salt loading""","""ECHEBASTAR""",null
2025-11-18,"""Mike""","""F6""",17:00:00,22:00:00,05:00:00,"""ALAKRANA""","""Salt loading""","""ECHEBASTAR""",null


In [10]:
forklift_salt = (
    salt.with_columns(pl.col("duration").str.to_time(format="%H:%M"))
    .filter(pl.col("date").dt.year().eq(2025))
    .filter(pl.col("operation_type").eq(pl.lit("Loading (Quay to Ship)")))
    .select(
        pl.col("date"),
        pl.col("vessel").cast(pl.Utf8).alias("vessel"),
        pl.col("start_time").alias("start_time"),
        pl.col("end_time").alias("end_time"),
        pl.col("duration").alias("duration"),
        pl.col("tonnage"),
    )
    
)

In [11]:
forklift_combined = forklift_salt_log.join(
    forklift_salt,
    left_on=["Date of Service", "Vessel/Client"],
    right_on=["date", "vessel"],
    how="full",
)

In [14]:
forklift_combined.filter(
    pl.col("vessel").is_not_null().and_(pl.col("Driver").is_null())
).collect()

SchemaError: invalid series dtype: expected `String`, got `time` for series with name `duration`

In [14]:
forklift_combined.filter(pl.col("vessel").is_null())

In [15]:
forklift_combined.filter(
    pl.col("vessel").is_not_null().and_(pl.col("Driver").is_not_null())
)

## "Overlapping" Record

In [16]:
add_datetime = forklift_df.with_columns(
    [
        pl.datetime(
            pl.col("Date of Service").dt.year(),
            pl.col("Date of Service").dt.month(),
            pl.col("Date of Service").dt.day(),
            pl.col("Time Out").dt.hour(),
            pl.col("Time Out").dt.minute(),
            pl.col("Time Out").dt.second(),
        ).alias("datetime_out"),
        pl.datetime(
            pl.col("Date of Service").dt.year(),
            pl.col("Date of Service").dt.month(),
            pl.col("Date of Service").dt.day(),
            pl.col("Time In").dt.hour(),
            pl.col("Time In").dt.minute(),
            pl.col("Time In").dt.second(),
        ).alias("datetime_in"),
    ]
)


# Method 1: Self-join approach to find overlaps
overlaps_self_join = (
    add_datetime.join(
        add_datetime.select(
            [
                pl.col("Date of Service").alias("date2"),
                pl.col("Driver").alias("driver2"),
                pl.col("datetime_out").alias("out2"),
                pl.col("datetime_in").alias("in2"),
                pl.col("Vessel/Client").alias("vessel2"),
                pl.col("Purpose").alias("purpose2"),
            ]
        ),
        how="cross",
    )
    .filter(
        (pl.col("Date of Service") == pl.col("date2"))
        & (pl.col("Driver") == pl.col("driver2"))
        & (pl.col("datetime_out") != pl.col("out2"))  # Different records
        & (
            # Check for overlap: start1 < end2 AND start2 < end1
            (pl.col("datetime_out") < pl.col("in2"))
            & (pl.col("out2") < pl.col("datetime_in"))
        )
    )
    .select(
        [
            "Date of Service",
            "Driver",
            "Time Out",
            "Time In",
            "Vessel/Client",
            "Purpose",
            pl.col("out2").dt.time().alias("Overlap Time Out"),
            pl.col("in2").dt.time().alias("Overlap Time In"),
            pl.col("vessel2").alias("Overlap Vessel/Client"),
            pl.col("purpose2").alias("Overlap Purpose"),
        ]
    )
    .sort(["Date of Service", "Driver", "Time Out"])
)


overlaps_self_join.collect()

Date of Service,Driver,Time Out,Time In,Vessel/Client,Purpose,Overlap Time Out,Overlap Time In,Overlap Vessel/Client,Overlap Purpose
date,str,time,time,str,str,time,time,str,str
2025-08-11,"""Versange""",10:41:00,10:45:00,"""Franche Terre""","""Ship supplies""",10:43:00,10:51:00,"""CCCS""","""unloading truck"""
2025-08-11,"""Versange""",10:43:00,10:51:00,"""CCCS""","""unloading truck""",10:41:00,10:45:00,"""Franche Terre""","""Ship supplies"""
2025-09-15,"""Yanic""",10:28:00,10:35:00,"""Txori Berri""","""Ship Supplies""",10:30:00,10:40:00,"""izurdia""","""Ship Supplies"""
2025-09-15,"""Yanic""",10:30:00,10:40:00,"""izurdia""","""Ship Supplies""",10:28:00,10:35:00,"""Txori Berri""","""Ship Supplies"""
2025-09-29,"""Versange""",13:06:00,17:30:00,"""Dolomieu""","""Salt loading""",13:16:00,13:23:00,"""Elai Alai""","""ship Supplies"""
…,…,…,…,…,…,…,…,…,…
2025-11-12,"""Mike""",09:14:00,09:32:00,"""Txori bat""","""ship Supplies""",09:00:00,11:17:00,"""Dolomieu""","""Salt loading"""
2025-11-12,"""Mike""",10:02:00,10:06:00,"""Dolomieu""","""ship Supplies""",09:00:00,11:17:00,"""Dolomieu""","""Salt loading"""
2025-11-12,"""Mike""",11:05:00,11:07:00,"""Txori bat""","""ship Supplies""",09:00:00,11:17:00,"""Dolomieu""","""Salt loading"""


In [21]:
overlaps_self_join.filter(pl.col("Date of Service").dt.month().eq(11)).collect().write_clipboard()

In [37]:
forklift_df.select(pl.col("Date of Service").value_counts().cast(pl.Utf8)).collect().write_clipboard()

In [12]:
forklift_df.with_columns(
    idx=pl.col("Vessel/Client")
    + pl.col("Date of Service").dt.to_string(format="%Y%m%d")
    + pl.col("Time Out").dt.to_string(format="%H%M")
).filter(pl.col("idx").is_duplicated().and_(pl.col("Date of Service").dt.month().eq(7))).drop("idx").collect()

Date of Service,Driver,Forklift No.,Time Out,Time In,Duration,Vessel/Client,Purpose,Invoiced in:,Column1
date,str,str,time,time,time,str,str,str,null
2025-07-17,"""Baptiste""","""F6""",13:35:00,13:40:00,00:05:00,"""Txori Toki""","""ship supplies""","""STO""",null
2025-07-17,"""Bryan""","""F5""",13:35:00,13:50:00,00:15:00,"""Txori Toki""","""shifting boat supplies""","""STO""",null


In [22]:
day_number = 25

In [24]:
forklift_df.filter(pl.col("Date of Service").dt.month().eq(7)).filter(pl.col("Vessel/Client").eq("Alakranatxu")).collect()

Date of Service,Driver,Forklift No.,Time Out,Time In,Duration,Vessel/Client,Purpose,Invoiced in:
date,str,str,time,time,time,str,str,str


In [23]:
forklift_df.filter(pl.col("Date of Service").dt.month().eq(7).and_(pl.col("Date of Service").dt.day().eq(day_number))).collect()



# .filter(pl.col("Driver").eq("Andy"))

# .filter(pl.col("Vessel/Client").eq("Aterpe Alai"))

Date of Service,Driver,Forklift No.,Time Out,Time In,Duration,Vessel/Client,Purpose,Invoiced in:
date,str,str,time,time,time,str,str,str
2025-07-25,"""Baptiste""","""F6""",17:10:00,17:11:00,00:01:00,"""Albacora Cuatro""","""unloading container""","""STO"""
2025-07-25,"""Baptiste""","""F6""",17:30:00,19:00:00,01:30:00,"""Albacora Cuatro""","""unloading container""","""STO"""
2025-07-25,"""Baptiste""","""F6""",17:45:00,21:30:00,03:45:00,"""Albacora Cuatro""","""unloading container""","""STO"""
2025-07-25,"""Bryan""","""F7""",17:39:00,19:30:00,01:51:00,"""Elai Alai""","""Salt loading""","""ECHEBASTAR"""


In [24]:
forklift.filter(pl.col("date").dt.year().eq(2025)).filter(pl.col("date").dt.month().eq(7).and_(pl.col("date").dt.day().eq(day_number))).collect()


# .filter(pl.col("customer").eq("ATERPE ALAI"))

day,date,start_time,end_time,duration,customer,invoiced_in,service_type,overtime_150,overtime_200,normal_hours
enum,date,time,time,time,str,str,str,i64,i64,i64
"""Fri""",2025-07-25,17:10:00,17:11:00,00:01:00,"""ALBACORA CUATRO""","""COMPAÑIA EUROPEA DE TUNIDOS,S.…","""Container Unloading""",1,0,0
"""Fri""",2025-07-25,17:30:00,19:00:00,01:30:00,"""ALBACORA CUATRO""","""COMPAÑIA EUROPEA DE TUNIDOS,S.…","""Container Unloading""",90,0,0
"""Fri""",2025-07-25,17:45:00,21:30:00,03:45:00,"""ALBACORA CUATRO""","""COMPAÑIA EUROPEA DE TUNIDOS,S.…","""Container Unloading""",225,0,0


In [21]:
forklift_df.join(
    forklift,
    left_on=["Date of Service", "Time Out", "Time In"],
    right_on=["date", "start_time", "end_time"],
    how="full",
).filter(pl.col("day").is_null()).sort("Date of Service").filter(pl.col("Date of Service").dt.month().eq(6)).collect()

Date of Service,Driver,Forklift No.,Time Out,Time In,Duration,Vessel/Client,Purpose,Invoiced in:,day,date,start_time,end_time,duration,customer,invoiced_in,service_type,overtime_150,overtime_200,normal_hours
date,str,str,time,time,time,str,str,str,enum,date,time,time,str,str,str,str,i64,i64,i64
2025-06-10,"""Baptiste""","""F6""",10:30:00,10:36:00,00:06:00,"""Draco""","""ship supplies""","""STO""",null,null,null,null,null,null,null,null,null,null,null
2025-06-10,"""Bryan ""","""F5""",16:08:00,19:40:00,03:32:00,"""Aterpe Alai""","""Salt loading""","""ECHEBASTAR""",null,null,null,null,null,null,null,null,null,null,null
2025-06-16,"""Bryan ""","""F5""",15:50:00,15:57:00,00:07:00,"""Alakrana""","""ship supplies""","""ECHEBASTAR""",null,null,null,null,null,null,null,null,null,null,null
2025-06-18,"""Bryan ""","""F4""",17:00:00,21:15:00,04:15:00,"""Txori berri""","""Salt loading""","""STO""",null,null,null,null,null,null,null,null,null,null,null
2025-06-22,"""Bryan ""","""F4""",08:17:00,09:23:00,01:06:00,"""Avel Vad""","""Salt loading""","""STO""",null,null,null,null,null,null,null,null,null,null,null
2025-06-24,"""Bryan ""","""F5""",19:15:00,22:00:00,02:45:00,"""JAI ALAI""","""Salt loading""","""STO""",null,null,null,null,null,null,null,null,null,null,null
2025-06-26,"""Bryan ""","""F5""",07:44:00,07:52:00,00:08:00,"""bernica""","""shifting staircase""","""SAPMER - UPSTREAM""",null,null,null,null,null,null,null,null,null,null,null
2025-06-26,"""Bryan ""","""F5""",17:47:00,22:00:00,04:13:00,"""Elai Alai""","""Salt loading""","""ECHEBASTAR""",null,null,null,null,null,null,null,null,null,null,null
